<a href="https://colab.research.google.com/github/NABI-SNU/book/blob/main/tutorials/Session_4_ReinforcementLearning/student/Tutorial1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/NABI-SNU/book/main/tutorials/Session_4_ReinforcementLearning/student/Tutorial1.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# Tutorial 1: The multi-armed bandit

**Session 4: Reinforcement Learning**

A *bandit* is a slot machine. Facing several machines whose payouts you do not know, every pull forces a choice: **exploit** the machine that has paid best so far, or **explore** another one that might be better.
The bandit is the explore–exploit dilemma in its purest form: there is only one state, so the only thing to learn is the value of each action.

By the end of this tutorial you will be able to:

- estimate action values incrementally from sampled rewards;
- implement ε-greedy and softmax action selection;
- explain why a purely greedy agent can lock onto a suboptimal action.

This tutorial follows Part 1 of the lecture and Chapter 2 of Sutton & Barto.

## Setup

In [ ]:
# @title Imports
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## 1. Four slot machines

As in the lecture, there are four machines **A–D**. Each pull returns a noisy reward around a fixed mean that the agent does not know.
Machine **B** pays reliably well, but machine **D** is better on average. Machine **A** loses money.

In [ ]:
ARMS = ["A", "B", "C", "D"]
TRUE_MEANS = np.array([-1.0, 0.5, 0.2, 1.0])  # D is the best arm on average
REWARD_SD = 1.0


def pull(arm, rng):
    """Sample one reward from the chosen arm (or arms, if `arm` is an array)."""
    return TRUE_MEANS[arm] + REWARD_SD * rng.standard_normal(np.shape(arm))


rng = np.random.default_rng(0)
samples = np.array([pull(np.full(2000, a), rng) for a in range(len(ARMS))])

fig, ax = plt.subplots(figsize=(6, 3))
ax.violinplot(samples.T, positions=range(len(ARMS)), showmeans=True)
ax.set_xticks(range(len(ARMS)), ARMS)
ax.axhline(0, color="0.7", lw=0.8)
ax.set_xlabel("machine")
ax.set_ylabel("reward per pull")
ax.set_title("Reward distributions (unknown to the agent)")
plt.show()

## 2. Learning action values from samples

The value of an action is its expected reward, $q_*(a) = \mathbb{E}[R \mid A = a]$. The agent keeps an estimate $Q(a)$ and, after the $n$-th reward $R_n$ from that action, nudges it toward the new sample:

$$Q_{n+1} = Q_n + \frac{1}{n}\left[R_n - Q_n\right].$$

This is the running average written as *old estimate + step size × error*, the same shape as every update in this session, including the TD update in Tutorial 2.

### Exercise: ε-greedy action selection

With probability $1-\varepsilon$ choose the action with the highest $Q$ (breaking ties at random); with probability $\varepsilon$ choose an action uniformly at random.

The function receives a batch of `Q` estimates with shape `(runs, arms)` so that many independent agents can be simulated at once. Try the TODO cell, then reveal the reference solution.

In [ ]:
# TODO: implement this function.
# def epsilon_greedy(Q, epsilon, rng):
#     """Return one action index per row of Q."""
#     ...

In [ ]:
def argmax_random_tie(Q, rng):
    """Row-wise argmax that breaks ties uniformly at random."""
    noise = rng.random(Q.shape) * 1e-9
    return np.argmax(Q + noise * (Q == Q.max(axis=1, keepdims=True)), axis=1)


def epsilon_greedy(Q, epsilon, rng):
    """Return one action index per row of Q."""
    runs, n_arms = Q.shape
    greedy = argmax_random_tie(Q, rng)
    random_actions = rng.integers(n_arms, size=runs)
    explore = rng.random(runs) < epsilon
    return np.where(explore, random_actions, greedy)

### Run many agents in parallel

Each run is a fresh agent that starts with $Q(a)=0$ for all machines and plays `steps` pulls. Averaging over many runs smooths out the luck of any single agent.

In [ ]:
def run_bandit(select_action, steps=1000, runs=2000, seed=0, **kwargs):
    """Simulate `runs` independent agents. Returns mean reward and P(best arm) per step."""
    rng = np.random.default_rng(seed)
    n_arms = len(TRUE_MEANS)
    Q = np.zeros((runs, n_arms))
    N = np.zeros((runs, n_arms))
    best = np.argmax(TRUE_MEANS)
    rewards = np.zeros(steps)
    optimal = np.zeros(steps)
    rows = np.arange(runs)
    for t in range(steps):
        kwargs_t = {k: (v(t) if callable(v) else v) for k, v in kwargs.items()}
        a = select_action(Q, rng=rng, **kwargs_t)
        r = pull(a, rng)
        N[rows, a] += 1
        Q[rows, a] += (r - Q[rows, a]) / N[rows, a]
        rewards[t] = r.mean()
        optimal[t] = (a == best).mean()
    return rewards, optimal


results = {eps: run_bandit(epsilon_greedy, epsilon=eps) for eps in (0.0, 0.01, 0.1)}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for eps, (rewards, optimal) in results.items():
    label = "greedy (ε = 0)" if eps == 0 else f"ε = {eps}"
    axes[0].plot(rewards, label=label)
    axes[1].plot(100 * optimal, label=label)
axes[0].set_ylabel("average reward")
axes[1].set_ylabel("% pulls on best machine (D)")
for ax in axes:
    ax.set_xlabel("pull")
    ax.legend()
fig.tight_layout()
plt.show()

```{admonition} Think!
- Why does the greedy agent plateau below the ε-greedy agents? Look at what happens the first time it gets a positive reward from machine B.
- ε = 0.1 finds machine D quickly but never chooses it more than about 92.5% of the time. Where does that ceiling come from?
- ε = 0.01 is slower at first. Which agent would you expect to be ahead after 10,000 pulls?
```

## 3. Softmax action preference

ε-greedy explores *blindly*: when it explores, a terrible machine is as likely as a promising one. A softmax agent instead chooses each action with probability that grows with its estimated value,

$$\pi(a) = \frac{e^{Q(a)/\tau}}{\sum_b e^{Q(b)/\tau}}.$$

The temperature $\tau$ controls the trade-off: large $\tau$ is close to uniform (more random), small $\tau$ is close to greedy.

### Exercise: softmax action selection

Return one sampled action per row of `Q`. Subtract the row maximum before exponentiating to avoid overflow. Try the TODO cell, then reveal the reference solution.

In [ ]:
# TODO: implement this function.
# def softmax_policy(Q, tau, rng):
#     """Sample one action per row of Q from the softmax distribution."""
#     ...

In [ ]:
def softmax_probabilities(Q, tau):
    z = (Q - Q.max(axis=1, keepdims=True)) / tau
    p = np.exp(z)
    return p / p.sum(axis=1, keepdims=True)


def softmax_policy(Q, tau, rng):
    """Sample one action per row of Q from the softmax distribution."""
    p = softmax_probabilities(Q, tau)
    u = rng.random((Q.shape[0], 1))
    return np.minimum((p.cumsum(axis=1) < u).sum(axis=1), Q.shape[1] - 1)

In [ ]:
Q_example = np.array([[-1.0, 0.5, 0.2, 1.0]])
fig, axes = plt.subplots(1, 3, figsize=(10, 2.6), sharey=True)
for ax, tau in zip(axes, (5.0, 0.5, 0.1)):
    ax.bar(ARMS, softmax_probabilities(Q_example, tau)[0], color="tab:purple")
    ax.set_title(f"τ = {tau}")
axes[0].set_ylabel("π(a)")
fig.suptitle("Softmax policy for Q = (-1, 0.5, 0.2, 1)")
fig.tight_layout()
plt.show()

In [ ]:
softmax_results = {tau: run_bandit(softmax_policy, tau=tau) for tau in (1.0, 0.3, 0.1)}

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(100 * results[0.1][1], color="0.5", ls="--", label="ε-greedy, ε = 0.1")
for tau, (_, optimal) in softmax_results.items():
    ax.plot(100 * optimal, label=f"softmax, τ = {tau}")
ax.set_xlabel("pull")
ax.set_ylabel("% pulls on best machine (D)")
ax.legend()
plt.show()

```{admonition} Think!
- Why does softmax with τ = 1.0 settle below 50% on machine D, even after its value estimates are accurate? Compute π(D) from the true means.
- At τ = 0.1, machine A is almost never pulled once its estimate is negative. Why is that an advantage over ε-greedy here?
```

## 4. Decaying exploration

Exploration is most valuable early, when estimates are poor. A common fix is to decay ε over time (the lecture's *epsilon decay*), for example $\varepsilon_t = \varepsilon_0 / (1 + t/k)$. `run_bandit` accepts a function of the step for any keyword argument.

In [ ]:
decay = run_bandit(epsilon_greedy, epsilon=lambda t: 0.3 / (1 + t / 50))

fig, ax = plt.subplots(figsize=(6, 3.5))
for eps in (0.01, 0.1):
    ax.plot(100 * results[eps][1], label=f"ε = {eps}")
ax.plot(100 * decay[1], label="decaying ε (0.3 → 0)")
ax.set_xlabel("pull")
ax.set_ylabel("% pulls on best machine (D)")
ax.legend()
plt.show()

## Summary

- The value of an action is an expected reward; it can be learned by repeatedly moving an estimate toward sampled rewards.
- A purely greedy agent can commit to the first action that looks good. Some exploration is needed to find better ones.
- ε-greedy explores uniformly; softmax explores in proportion to estimated value, with the temperature setting how random it is.

The bandit has no notion of *later*: each pull is scored only by its immediate reward. Tutorial 2 adds time, so that an early event (a cue) can acquire value from a reward that comes after it.

## References

- Sutton, R. S. & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.), Chapter 2. MIT Press.